In [1]:
import os
import random
import copy
import numpy as np
import pandas as pd
from PIL import Image, UnidentifiedImageError
from sklearn.model_selection import train_test_split
from tqdm import tqdm

import subprocess
subprocess.run(["pip", "install", "open-clip-torch"], check=True)
import open_clip

import torch
import torch.nn as nn
import torch.optim as optim
from torch.cuda.amp import GradScaler, autocast
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torchvision.transforms as transforms

torch.backends.cudnn.benchmark = True

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 50.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.0 MB/s eta 0:00:00
Using device: cuda


In [2]:
BASE_INPUT      = "/kaggle/input/competitions/cse-281-spring-26-scene-style-classification"
BASE            = os.path.join(BASE_INPUT, "StyleClassificationIndoors/StyleClassificationIndoors")
SAMPLE_SUB_PATH = os.path.join(BASE_INPUT, "sample_submission.csv")
TRAIN_DIR       = os.path.join(BASE, "train")
TEST_DIR        = os.path.join(BASE, "test")
BATCH_SIZE      = 16

class_names   = sorted(os.listdir(TRAIN_DIR))
class_mapping = {name: idx for idx, name in enumerate(class_names)}
NUM_CLASSES   = len(class_names)
print(f"Classes ({NUM_CLASSES}):", class_mapping)

all_paths, all_labels = [], []
for class_name, class_id in class_mapping.items():
    class_dir = os.path.join(TRAIN_DIR, class_name)
    if not os.path.isdir(class_dir): continue
    for fname in os.listdir(class_dir):
        if fname.lower().endswith((".jpg", ".jpeg", ".png")):
            all_paths.append(os.path.join(class_dir, fname))
            all_labels.append(class_id)

train_paths, val_paths, train_labels, val_labels = train_test_split(
    all_paths, all_labels, test_size=0.2, random_state=SEED, stratify=all_labels
)

print(f"Train: {len(train_paths)} | Val: {len(val_paths)}")

Classes (17): {'asian': 0, 'boho': 1, 'coastal': 2, 'contemporary': 3, 'craftsman': 4, 'eclectic': 5, 'farmhouse': 6, 'french-country': 7, 'industrial': 8, 'mediterranean': 9, 'minimalist': 10, 'modern': 11, 'scandinavian': 12, 'shabby-chic-style': 13, 'southwestern': 14, 'tropical': 15, 'victorian': 16}
Train: 10530 | Val: 2633


In [3]:
INPUT_SIZE = 224
EVA_MEAN = [0.48145466, 0.4578275,  0.40821073]
EVA_STD  = [0.26862954, 0.26130258, 0.27577711]

train_transform = transforms.Compose([
    transforms.Resize((INPUT_SIZE, INPUT_SIZE)),
    transforms.RandAugment(num_ops=2, magnitude=12),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(EVA_MEAN, EVA_STD)
])

val_transform = transforms.Compose([
    transforms.Resize((INPUT_SIZE, INPUT_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(EVA_MEAN, EVA_STD)
])

class SceneDataset(Dataset):
    def __init__(self, image_paths, labels, transform=None):
        self.image_paths = image_paths
        self.labels      = labels
        self.transform   = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        try:
            image = Image.open(self.image_paths[idx]).convert("RGB")
            if self.transform:
                image = self.transform(image)
            return image, self.labels[idx]
        except (UnidentifiedImageError, IOError, OSError, TypeError):
            return torch.zeros((3, INPUT_SIZE, INPUT_SIZE)), 0

class_counts   = np.bincount(train_labels)
sample_weights = [1.0 / class_counts[l] for l in train_labels]
sampler        = WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True)

train_loader = DataLoader(
    SceneDataset(train_paths, train_labels, train_transform),
    batch_size=BATCH_SIZE, sampler=sampler,
    num_workers=4, pin_memory=True,
    persistent_workers=True, prefetch_factor=2
)
val_loader = DataLoader(
    SceneDataset(val_paths, val_labels, val_transform),
    batch_size=BATCH_SIZE, shuffle=False,
    num_workers=4, pin_memory=True,
    persistent_workers=True, prefetch_factor=2
)
print(f"Dataset ready | Train: {len(train_paths)} | Val: {len(val_paths)}")

Dataset ready | Train: 10530 | Val: 2633


In [4]:
WEIGHTS_PATH = "/kaggle/input/datasets/janagoharyy/evadaataset/eva_clip_weights.pt"
EMBED_DIM    = 768

print("Building EVA02-L-14...")
eva_model, _, _ = open_clip.create_model_and_transforms('EVA02-L-14', pretrained=None)
checkpoint = torch.load(WEIGHTS_PATH, map_location='cpu', weights_only=False)
eva_model.load_state_dict(checkpoint, strict=True)
backbone = eva_model.visual
print("Weights loaded! Sample weight sum:", sum(p.sum().item() for p in eva_model.visual.parameters()))
print("EVA02-L-14 loaded")

class EVAClassifier(nn.Module):
    def __init__(self, backbone, embed_dim, num_classes):
        super().__init__()
        self.backbone = backbone
        self.classifier = nn.Sequential(
            nn.Linear(embed_dim, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(512, num_classes)
)

    def forward(self, x):
        features = self.backbone(x)
        return self.classifier(features)

model = EVAClassifier(backbone, EMBED_DIM, NUM_CLASSES).to(device)
print(f"Model ready → {NUM_CLASSES} classes | device: {device}")

Building EVA02-L-14...


Weights loaded! Sample weight sum: 191415.07737368345
EVA02-L-14 loaded
Model ready → 17 classes | device: cuda


In [5]:
similarity_matrix = torch.zeros(17, 17)

similar_pairs = [
    (10, 12),  # minimalist ↔ scandinavian
    (11, 3),   # modern ↔ contemporary
    (6, 4),    # farmhouse ↔ craftsman
    (13, 7),   # shabby-chic ↔ french-country
    (2, 15),   # coastal ↔ tropical
    (1, 5),    # boho ↔ eclectic
    (9, 7),    # mediterranean ↔ french-country
    (8, 11),   # industrial ↔ modern
    (16, 13),  # victorian ↔ shabby-chic
]
for i, j in similar_pairs:
    similarity_matrix[i][j] = 1.0
    similarity_matrix[j][i] = 1.0

similarity_matrix = similarity_matrix.to(device)

class SimilarityAwareLoss(nn.Module):
    def __init__(self, similarity_matrix, similarity_penalty=1.5):
        super().__init__()
        self.similarity_matrix  = similarity_matrix
        self.similarity_penalty = similarity_penalty

    def forward(self, outputs, targets):
        ce_loss   = nn.functional.cross_entropy(outputs, targets, reduction='none')
        predicted = outputs.argmax(dim=1)
        penalties = self.similarity_matrix[targets, predicted]
        penalty   = 1.0 + (penalties > 0).float() * (self.similarity_penalty - 1.0)
        return (ce_loss * penalty).mean()

criterion = SimilarityAwareLoss(similarity_matrix, similarity_penalty=1.5)
print("✅ SimilarityAwareLoss ready!")

def run_epoch(model, loader, optimizer, criterion, device, scaler, is_train):
    model.train() if is_train else model.eval()
    running_loss = running_correct = total_samples = 0

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        if is_train:
            optimizer.zero_grad(set_to_none=True)
            with autocast():
                outputs = model(images)
                loss    = criterion(outputs, labels)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
        else:
            with torch.no_grad(), autocast():
                outputs = model(images)
                loss    = criterion(outputs, labels)

        running_loss    += loss.item() * images.size(0)
        running_correct += (outputs.argmax(1) == labels).sum().item()
        total_samples   += images.size(0)

    return running_loss / total_samples, running_correct / total_samples

print("✅ run_epoch ready")

✅ SimilarityAwareLoss ready!
✅ run_epoch ready


In [6]:
# All parameters trainable from the start — no freezing
for param in model.parameters():
    param.requires_grad = True

optimizer = optim.Adam([
    {'params': model.backbone.parameters(), 'lr': 5e-7},
    {'params': model.classifier.parameters(), 'lr': 1e-4},
], weight_decay=1e-3)

scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=6)
scaler    = GradScaler()

NUM_EPOCHS     = 6
best_val_acc   = 0.0
best_model_wts = copy.deepcopy(model.state_dict())

print("="*50)
print("Training — 6 epochs, full model, 224px")
print("="*50)

for epoch in range(NUM_EPOCHS):
    t_loss, t_acc = run_epoch(model, train_loader, optimizer, criterion, device, scaler, True)
    v_loss, v_acc = run_epoch(model, val_loader,   None,      criterion, device, scaler, False)
    scheduler.step()
    print(f"Epoch {epoch+1}/{NUM_EPOCHS} | Train Acc: {t_acc*100:.2f}% | Val Acc: {v_acc*100:.2f}%")
    if v_acc > best_val_acc:
        best_val_acc   = v_acc
        best_model_wts = copy.deepcopy(model.state_dict())
        torch.save(model.state_dict(), "/kaggle/working/best_eva_model.pt")
        print(f"  ✅ Best saved! Val Acc: {best_val_acc*100:.2f}%")

model.load_state_dict(best_model_wts)
print(f"\nDone. Best Val Acc: {best_val_acc*100:.2f}%")

Training — 6 epochs, full model, 224px


/tmp/ipykernel_23/1244977771.py:11: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler    = GradScaler()
/tmp/ipykernel_23/2623590378.py:46: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
/tmp/ipykernel_23/2623590378.py:55: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.no_grad(), autocast():


Epoch 1/6 | Train Acc: 46.02% | Val Acc: 59.36%
  ✅ Best saved! Val Acc: 59.36%
Epoch 2/6 | Train Acc: 58.47% | Val Acc: 60.12%
  ✅ Best saved! Val Acc: 60.12%
Epoch 3/6 | Train Acc: 59.80% | Val Acc: 61.83%
  ✅ Best saved! Val Acc: 61.83%
Epoch 4/6 | Train Acc: 61.25% | Val Acc: 61.75%
Epoch 5/6 | Train Acc: 61.98% | Val Acc: 61.79%
Epoch 6/6 | Train Acc: 63.70% | Val Acc: 61.75%

Done. Best Val Acc: 61.83%


In [7]:
base_transform_tta = transforms.Compose([
    transforms.Resize((INPUT_SIZE, INPUT_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(EVA_MEAN, EVA_STD)
])

tta_transform = transforms.Compose([
    transforms.Resize((INPUT_SIZE, INPUT_SIZE)),
    transforms.RandomHorizontalFlip(p=1.0),
    transforms.RandomRotation(degrees=10),
    transforms.ToTensor(),
    transforms.Normalize(EVA_MEAN, EVA_STD)
])

def tta_predict(model, img_path, n_aug=3):  # reduced from 5 → 3 for speed
    image   = Image.open(img_path).convert('RGB')
    outputs = torch.softmax(model(base_transform_tta(image).unsqueeze(0).to(device)), dim=1)
    for _ in range(n_aug):
        outputs += torch.softmax(model(tta_transform(image).unsqueeze(0).to(device)), dim=1)
    return outputs / (n_aug + 1)

submission_df  = pd.read_csv(SAMPLE_SUB_PATH)
label_col_name = submission_df.columns[1]

model.eval()
predictions = []
print(f"Processing {len(submission_df)} images...")

with torch.no_grad():
    for image_id in submission_df.iloc[:, 0]:
        img_path = os.path.join(TEST_DIR, str(image_id))
        try:
            outputs = tta_predict(model, img_path, n_aug=3)
            predictions.append(outputs.argmax(1).item())
        except (UnidentifiedImageError, FileNotFoundError):
            predictions.append(0)

submission_df[label_col_name] = predictions
submission_df.to_csv("/kaggle/working/submission_eva.csv", index=False)
print("✅ submission_eva.csv saved!")
print(submission_df.head())

Processing 5482 images...


/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


✅ submission_eva.csv saved!
         ImageName  ClassLabel
0  testimage_1.jpg          14
1  testimage_2.jpg          10
2  testimage_3.jpg           1
3  testimage_4.jpg          10
4  testimage_5.jpg           1
